## MLP with feature engineering (careful split for use in LSTM later)

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, classification_report, precision_score
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset  # for saving after removing features
import kagglehub

# check if GPU available
gpu_av=torch.cuda.is_available()

# for reproducibility
SEED = 42
torch.manual_seed(SEED)
print("GPU available:", gpu_av)
if gpu_av:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"CPU cores: {os.cpu_count()}")

In [ ]:
# SAME functions as Project.ipynb latest version

def download_dataset(
    year_start,
    year_end_exd,  # end year (excluded)
    origin_path="flnny123/mfddmulti-modal-flight-delay-dataset/versions/4",
    mode="tabular",  # or "sequential" for pre-made chains
    output_dir_seq="data/chain/",
):

    dest_paths = []
    for year in range(year_start, year_end_exd):
        print(f"Downloading year {year} data...")
        if mode == "tabular":  # tabular dataset download
            origin_path_year = (
                "Aeolus/Flight_Tab/flight_with_weather_" + str(year) + ".csv"
            )
            dest_path_year = kagglehub.dataset_download(
                origin_path, path=origin_path_year
            )
            dest_paths.append(dest_path_year)

        # dest_path_year = dest_path+'flight_with_weather_'+str(year)+'.csv' # destination path
        elif mode == "sequential":  # sequential (chains) dataset download
            for split in ["train", "val", "test"]:
                if year == 2024:
                    origin_path_year_split = f"Aeolus/Flight_chain/chain_data_{year}/flight_chain_{split}_{year}.pt"
                # different naming convention in original dataset
                else:
                    origin_path_year_split = f"Aeolus/Flight_chain/chain_data_{year}/{split}_flight_chain_{year}.pt"

                final_path = os.path.join(
                    output_dir_seq + str(year), f"{split}_flight_chain_{year}.pt"
                )
                if os.path.exists(final_path):
                    print(f"Path {final_path} already exists! Skipping it")
                    continue

                dest_path_year = kagglehub.dataset_download(
                    origin_path, path=origin_path_year_split
                )
                os.makedirs(output_dir_seq + str(year), exist_ok=True)
                shutil.move(dest_path_year, final_path)
                dest_path_year = final_path
                dest_paths.append(final_path)

                print(f"(File(s) available at {dest_path_year}).")

    return dest_paths


def load_dataset_pytorch(year_start, year_end, file_path="data/chain/"):
    """
    Load sequential chain datasets for years in [year_start, year_end) and merge
    all years into a single dataset per split (train, val, test).
    """
    split_types = ["train", "val", "test"]
    loaded_data = {split: [] for split in split_types}  # store lists of tensors

    for year in range(year_start, year_end):
        for split in split_types:
            full_file_path = file_path + f"{year}/{split}_flight_chain_{year}.pt"
            dataset = torch.load(full_file_path, weights_only=False)
            # n_samples_to_inspect = 3

            # for i in range(n_samples_to_inspect):
            #     sample = dataset[i]
            #     print(f"--- Sample {i} ---")
            #     for j, tensor in enumerate(sample):
            #         print(f"  tensors[{j}] shape: {tensor.shape}, dtype: {tensor.dtype}")
            #         print(f"  tensors[{j}] values:\n{tensor}\n")
            #     print("=" * 60)

            # Slice dense tensor to remove FLIGHTS (last column)
            dense = dataset.tensors[0]  # [N, seq_len, 7]
            dense = dense[:, :, :-1].clone()  # [N, seq_len, 6]

            # Rebuild dataset (other tensors unchanged)
            processed = TensorDataset(dense, *dataset.tensors[1:])
            loaded_data[split].append(processed)
            print(f"--- Read file: (split: {split}, year: {year}) ---")

    # Concatenate all years for each split
    merged = {}
    for split in split_types:
        # Gather all tensors from each dataset in the list
        all_dense = torch.cat([ds.tensors[0] for ds in loaded_data[split]], dim=0)
        all_sparse = torch.cat([ds.tensors[1] for ds in loaded_data[split]], dim=0)
        all_labels = torch.cat([ds.tensors[2] for ds in loaded_data[split]], dim=0)
        all_lens = torch.cat([ds.tensors[3] for ds in loaded_data[split]], dim=0)
        all_delays = torch.cat([ds.tensors[4] for ds in loaded_data[split]], dim=0)

        merged[split] = TensorDataset(
            all_dense, all_sparse, all_labels, all_lens, all_delays
        )
        print(f"--- Merged {split}: {all_dense.shape[0]} samples across years ---")

    return merged



In [ ]:
year_st = 2022
file_path = download_dataset(
    year_start=year_st, year_end_exd=year_st + 2, mode="sequential"
)  # or "sequential" for pre-made chains)


loaded_data=load_dataset_pytorch(year_st, year_st+2)

In [ ]:
# On targets
train_dataset = loaded_data["train"]
all_delays = train_dataset.tensors[4]      # [N, seq_len, 2]
all_valid_lens = train_dataset.tensors[3]  # [N]

seq_len = all_delays.shape[1]
mask = torch.arange(seq_len).unsqueeze(0) < all_valid_lens.unsqueeze(1)  # [N, seq_len]

valid_delays = all_delays[mask].float()  # [total_valid_steps, 2] -- only as big as valid entries, no dataset copy

delay_mean = valid_delays.mean(dim=0)  # shape [2]
delay_std = valid_delays.std(dim=0)    # shape [2]

print(f"Delay mean (ARR, DEP): {delay_mean}")
print(f"Delay std (ARR, DEP): {delay_std}")



In [ ]:
# Scale/unscale helpers (operate on tensors on-the-fly, no dataset copy) ---
def scale_tensors(targets, mean, std):
    return (targets - mean.to(targets.device)) / std.to(targets.device)

def unscale_tensors(scaled, mean, std):
    return scaled * std.to(scaled.device) + mean.to(scaled.device)

delay_threshold_scaled = scale_tensors(torch.tensor([15.0, 15.0]), mean=delay_mean, std=delay_std)  # 15-min threshold in SCALED units
print(delay_threshold_scaled)



In [ ]:
# On dense features
all_dense = train_dataset.tensors[0]  # [N, seq_len, 6]
valid_dense = all_dense[mask].float()  # [total_valid_steps, 6], reuses same mask as before

dense_mean = valid_dense.mean(dim=0)  # [6]
dense_std = valid_dense.std(dim=0)    # [6]

print(dense_mean)
print(dense_std)

In [ ]:
# checks

N_train = len(loaded_data["train"])
N_val = len(loaded_data["val"])
N_test = len(loaded_data["test"])

N_chains = N_train + N_val + N_test

print(f"Total number of flight chains: {N_chains}")
print(f"Train samples: {N_train} ({N_train*100/N_chains:.2f}%)")
print(f"Validation samples: {N_val} ({N_val*100/N_chains:.2f}%)")
print(f"Test samples: {N_test} ({N_test*100/N_chains:.2f}%)")

# 1. Extract the first sample from the training dataset
sample = loaded_data["train"][0]

print(f"Sample Type: {type(sample)}")
print(f"Total components in the sample tuple: {len(sample)}\n")
print("-" * 60)

# 2. Unpack each component of the 5-element tuple into properly named variables
dense_feat, sparse_feat, labels, valid_lens, delays = sample

# 3. Print details and relative values for each component with descriptive labels based on source code
print("1. Dense Features (Continuous / Meteorological Features):")
print(f"   - Shape: {dense_feat.shape} (Sequence Length x 6)")
print(
    f"   - Cols: ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD']"
)
print(f"    - Values:\n{dense_feat}\n")

print("2. Sparse Features (Categorical / Temporal Features):")
print(f"   - Shape: {sparse_feat.shape} (Sequence Length x 8)")
print(
    f"   - Cols: "
)['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX', 'OP_CARRIER', 'OP_CARRIER_FL_NUM']
print(f"    - Values:\n{sparse_feat}\n")

print("3. Binary Labels (Flight Delay Indicators > 15 mins):")
print(f"   - Shape: {labels.shape} (Sequence Length x 2)")
print(f"   - [(ARR_DELAY > 15), (DEP_DELAY > 15)]")
print(f"   - Values:\n{labels}\n")

print("4. Valid Sequence Lengths (Metadata):")
print(
    f"   - Description: Effective number of valid flights in the chain before padding"
)
print(f"   - Shape: {valid_lens.shape}")
print(f"   - Values: {valid_lens}\n")

print("5. Raw Delays (Ground Truth):")
print(f"   - Shape: {delays.shape} (Sequence Length x 2)")
print(f"   - Cols: ['ARR_DELAY', 'DEP_DELAY']")
print(f"   - Values:\n{delays}")
print("-" * 60)

In [ ]:
# new tensor order: FL_YEAR, FL_DAY, 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', o lat, o long, d lat, d long, FL_WEEK, le 10 ingegnerizzate

In [ ]:
# features used in MLP: (32):
# 
# 'FL_DAY',  # day of the month - missing

# 'FL_WEEK', # week of the year - missing


# 'dep_hour_sin', 'dep_hour_cos', 'arr_hour_sin', 'arr_hour_cos', 'dow_sin', # calc from others
# 'dow_cos', 'month_sin', 'month_cos', 'great_circle_km', 'origin_congestion_2h',
# 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', # can be converted
# # 'CRS_ELAPSED_TIME', 'FLIGHTS', can be deduced
# #  'O_LATITUDE', 'O_LONGITUDE', 'D_LATITUDE', 'D_LONGITUDE',  # can be deduced
#  'FL_YEAR', # can be deduced
# 'FL_MONTH', 
# 'ORIGIN_INDEX', 'DEST_INDEX',
# 'O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD',
# 'OP_CARRIER', 'OP_CARRIER_FL_NUM',

In [ ]:
# features in chains:
# 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR',

# 'MONTH', 
# 'ORIGIN_INDEX', 'DEST_INDEX',
# 'OP_CARRIER', 'OP_CARRIER_FL_NUM'
# 'O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD'



### To avoid leaking in training / test split: from chain dataset, get individual flights as a dataframe (with engineered features) for MLP training



In [ ]:
def convert_chains_to_csv():

    


### Load data and train (same as MLP / data pipelines notebooks)

### Save model weights to be reused exactly the same in LSTM + MLP embedding single flight's features